# Assessment notebook v2 — one student/question per row
Reference schema: Student, Question, Question text, Rubric answer/title (verbatim),
Result, Points earned, Points possible, Points lost, Score (%).
Also accepts StudentName,Q1Text,A1Label,A1 with optional max_score.
One assessment per CSV. Student names identify students; use unique anonymous IDs
where duplicate names exist. Exact question IDs/text must match across papers.
No PDF extraction or grading is performed here.

Unknown maxima: retain raw rows and descriptive statistics, skip normalized fitting.
Zero-max, extra-credit and inconsistent-max questions are excluded from fitting.
Expected points = sum(max_points_j * fitted_expected_fraction_ij) on modeled items.
This is a regularized mean model, not a corrected grade or a partial-credit IRT
category model. Embeddings only support similarity retrieval; no assumption that
similar questions have the same difficulty. Review OCR and rubric text upstream.

Convenience: import this notebook and Run All, or paste codeblocks.py into ONE
Python notebook cell. No installation needed for scoring; embeddings use the
Databricks SDK. Default blank csv_path uses synthetic demo data.


In [0]:
# Run with no path for a synthetic preview, or set csv_path to your volume CSV.
import json
from pathlib import Path
try:
    dbutils
except NameError:
    CSV_PATH = ''
    EMBED_QUESTIONS = False
    RETURN_JSON = False
    EMBEDDING_ENDPOINT = 'databricks-gte-large-en'
else:
    for name,default in [('csv_path',''),('embed_questions','true'),('return_json','false'),('embedding_endpoint','databricks-gte-large-en')]:
        dbutils.widgets.text(name,default)
    CSV_PATH=dbutils.widgets.get('csv_path').strip()
    EMBED_QUESTIONS=dbutils.widgets.get('embed_questions').lower()=='true'
    RETURN_JSON=dbutils.widgets.get('return_json').lower()=='true'
    EMBEDDING_ENDPOINT=dbutils.widgets.get('embedding_endpoint').strip()


In [0]:
"""Dependency-free, regularized Rasch analysis of binary item responses.

Run: python analyze_irt.py student_responses.csv --out results
Blank cells are missing; only literal 0 and 1 are valid observed responses.
This is joint MAP estimation, not marginal/conditional maximum likelihood.
"""

import argparse
import csv
import json
import math
from pathlib import Path


def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    e = math.exp(z)
    return e / (1 + e)


def softplus(z):
    return max(z, 0) + math.log1p(math.exp(-abs(z)))


def load_responses(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))
    if len(rows) < 3 or len(rows[0]) < 3:
        raise ValueError("Provide a header, at least 2 students and 2 questions.")
    header = [s.strip() for s in rows[0]]
    if header[0] != "student_id":
        raise ValueError("First column must be student_id.")
    items = header[1:]
    if any(not s for s in items) or len(set(items)) != len(items):
        raise ValueError("Question names must be nonempty and unique.")
    ids, data = [], []
    for line, row in enumerate(rows[1:], 2):
        if len(row) != len(header):
            raise ValueError(f"Row {line}: wrong number of columns.")
        sid = row[0].strip()
        if not sid or sid in ids:
            raise ValueError(f"Row {line}: empty or duplicate student_id.")
        values = [v.strip() for v in row[1:]]
        if any(v not in ("", "0", "1") for v in values):
            raise ValueError(f"Row {line}: responses must be 0, 1, or blank.")
        ids.append(sid)
        data.append([None if v == "" else int(v) for v in values])
    if any(all(v is None for v in row) for row in data):
        raise ValueError("Every student needs at least one observed response.")
    if any(all(row[j] is None for row in data) for j in range(len(items))):
        raise ValueError("Every question needs at least one observed response.")
    return ids, items, data


def fit_rasch(data, theta_sd=1.0, difficulty_sd=2.0, max_sweeps=2000):
    """Minimize Bernoulli negative log likelihood plus Gaussian penalties.

    p_ij = sigmoid(theta_i - b_j)
    theta_i ~ Normal(0, theta_sd**2), b_j ~ Normal(0, difficulty_sd**2).
    Proper zero-centered priors anchor the otherwise unidentified location.
    Discrimination is fixed at 1. Coordinate Newton steps use backtracking.
    """
    n, m = len(data), len(data[0])
    theta, difficulty = [0.0] * n, [0.0] * m
    students = [[(j, x) for j, x in enumerate(row) if x is not None]
                for row in data]
    questions = [[(i, row[j]) for i, row in enumerate(data)
                  if row[j] is not None] for j in range(m)]

    def update(value, observations, offsets, precision, is_question):
        def loss(v):
            total = precision * v * v / 2
            for k, x in observations:
                z = offsets[k] - v if is_question else v - offsets[k]
                total += softplus(z) - x * z
            return total
        gradient = precision * value
        curvature = precision
        for k, x in observations:
            z = offsets[k] - value if is_question else value - offsets[k]
            p = sigmoid(z)
            gradient += x - p if is_question else p - x
            curvature += p * (1 - p)
        step, old_loss = gradient / curvature, loss(value)
        for _ in range(60):
            candidate = value - step
            if loss(candidate) <= old_loss + 1e-12:
                return candidate
            step *= 0.5
        raise RuntimeError("Line search failed.")

    tp, bp = 1 / theta_sd**2, 1 / difficulty_sd**2
    for sweep in range(1, max_sweeps + 1):
        for i in range(n):
            theta[i] = update(theta[i], students[i], difficulty, tp, False)
        for j in range(m):
            difficulty[j] = update(difficulty[j], questions[j], theta, bp, True)
        gt = [tp * t for t in theta]
        gb = [bp * b for b in difficulty]
        for i, row in enumerate(data):
            for j, x in enumerate(row):
                if x is not None:
                    residual = sigmoid(theta[i] - difficulty[j]) - x
                    gt[i] += residual
                    gb[j] -= residual
        max_gradient = max(abs(g) for g in gt + gb)
        if max_gradient < 1e-7:
            return theta, difficulty, sweep, max_gradient
    raise RuntimeError(f"Did not converge after {max_sweeps} sweeps.")


"""Long-form assessment analysis. Standard library only; no extraction code."""
import csv
import io
import math
from collections import Counter, defaultdict


def parse_csv(text):
    reader = csv.DictReader(io.StringIO(text.lstrip('\ufeff')))
    rows = []
    seen = set()
    def choose(r, *keys):
        return next((str(r[k]).strip() for k in keys if r.get(k) is not None and str(r[k]).strip()), '')
    def number(s, label):
        if not s: return None
        try: value = float(s)
        except ValueError: raise ValueError(f'{label}: invalid number')
        if not math.isfinite(value) or value < 0: raise ValueError(f'{label}: expected finite nonnegative points')
        return value
    for line, r in enumerate(reader, 2):
        if None in r: raise ValueError(f'Row {line}: extra CSV fields; quote commas in text')
        student = choose(r, 'Student', 'StudentName', 'student_id')
        question = choose(r, 'Question text', 'Q1Text', 'question_text')
        qid = choose(r, 'Question', 'question_id') or question
        if not student or not question: raise ValueError(f'Row {line}: student and question text required')
        key = (student, qid)
        if key in seen: raise ValueError(f'Row {line}: duplicate student/question; use one assessment per CSV')
        seen.add(key)
        earned = number(choose(r, 'Points earned', 'A1', 'score'), f'Row {line} earned')
        possible = number(choose(r, 'Points possible', 'max_score'), f'Row {line} possible')
        rows.append(dict(student=student, question_id=qid, question_text=question,
            label=choose(r, 'Result', 'A1Label', 'answer_label'),
            rubric_title=choose(r, 'Rubric answer/title (verbatim)', 'rubric_title'),
            earned=earned, possible=possible))
    if not rows: raise ValueError('CSV contains no rows')
    if len(rows)>100000: raise ValueError('Prototype limit: 100,000 rows')
    return rows


def analyze(rows, fit_rasch, embeddings=None):
    students = sorted({r['student'] for r in rows})
    questions = list(dict.fromkeys(r['question_id'] for r in rows))
    si = {s:i for i,s in enumerate(students)}
    qi = {q:j for j,q in enumerate(questions)}
    byq = defaultdict(list)
    bys = defaultdict(list)
    warnings = []
    for r in rows: byq[r['question_id']].append(r); bys[r['student']].append(r)
    for q, records in byq.items():
        if len({r['question_text'] for r in records}) != 1:
            raise ValueError(f'Question {q}: inconsistent text; reconcile extraction before modeling')
    text = [byq[q][0]['question_text'] for q in questions]
    X = [[None]*len(questions) for _ in students]
    max_points = []
    excluded = []
    for q in questions:
        maxima = {r['possible'] for r in byq[q] if r['possible'] is not None}
        valid_max = len(maxima)==1 and next(iter(maxima))>0
        p = next(iter(maxima)) if valid_max else None
        # Missing maximum is not filled from another student's record.
        if p is not None and any(r['earned'] is not None and r['earned']>p for r in byq[q]):
            p = None
        max_points.append(p)
        if p is None: excluded.append(q)
        for r in byq[q]:
            if p is not None and r['earned'] is not None and r['possible']==p:
                X[si[r['student']]][qi[q]] = r['earned']/p
    eligible_q=[j for j in range(len(questions)) if any(row[j] is not None for row in X)]
    eligible_s=[i for i in range(len(students)) if any(X[i][j] is not None for j in eligible_q)]
    mu = {}; theta_by_student = {}; difficulty_by_q = {}
    if len(eligible_q)>=2 and len(eligible_s)>=2:
        # Reject disconnected student/item graphs: separate components have no shared scale.
        visited_s={eligible_s[0]}; visited_q=set()
        while True:
            nq={j for i in visited_s for j in eligible_q if X[i][j] is not None}
            ns={i for j in nq for i in eligible_s if X[i][j] is not None}
            if ns==visited_s and nq==visited_q: break
            visited_s,visited_q=ns,nq
        if len(visited_s)!=len(eligible_s) or len(visited_q)!=len(eligible_q):
            warnings.append('Model skipped: disconnected student/question groups need shared items.')
        elif len(eligible_q)*len(eligible_s)>2000000:
            warnings.append('Model skipped: driver-only prototype exceeds 2 million cells.')
        else:
            matrix=[[X[i][j] for j in eligible_q] for i in eligible_s]
            t,b,sweeps,gradient=fit_rasch(matrix)
            for a,i in enumerate(eligible_s):
                theta_by_student[students[i]]=t[a]
                for c,j in enumerate(eligible_q):
                    z=t[a]-b[c]
                    mu[(i,j)]=1/(1+math.exp(-z))
            difficulty_by_q={questions[j]:b[c] for c,j in enumerate(eligible_q)}
    else: warnings.append('Model skipped: at least two students and two scored questions are needed.')
    if excluded: warnings.append('Questions excluded from normalized model (missing/inconsistent maxima, zero maximum, or extra credit): '+', '.join(excluded))
    warnings += ['Fractional-logit mean model with Gaussian regularization; not a partial-credit IRT category likelihood.',
        'Expected score is model-based, not a corrected grade. No uncertainty intervals.',
        'Expected totals cover only the modeled question subset; missing responses are omitted from fitting.',
        'Rubric labels are preserved literally; no automatic misconception or severity inference.',
        'Question embeddings drive related-question summaries, not assumed difficulty or proficiency.']
    if len(students)<30: warnings.append('Very small class: treat model estimates as illustrative only.')
    student_out=[]
    for i,s in enumerate(students):
        graded=[r for r in bys[s] if r['earned'] is not None]
        predicted=[j for j in eligible_q if (i,j) in mu]
        student_out.append(dict(student=s, observed_points=sum(r['earned'] for r in graded),
            graded_questions=len(graded), proficiency=theta_by_student.get(s),
            estimated_true_score=sum(mu[i,j]*max_points[j] for j in predicted) if predicted else None,
            modeled_max_points=sum(max_points[j] for j in predicted) if predicted else None,
            modeled_questions=len(predicted)))
    question_out=[]
    for j,q in enumerate(questions):
        graded=[r for r in byq[q] if r['earned'] is not None]
        normalized=[row[j] for row in X if row[j] is not None]
        vals=Counter(r['earned'] for r in graded)
        question_out.append(dict(question_id=q, question_text=text[j], graded=len(graded),
            mean_points=sum(r['earned'] for r in graded)/len(graded) if graded else None,
            possible=max_points[j], mean_fraction=sum(normalized)/len(normalized) if normalized else None,
            difficulty=difficulty_by_q.get(q), score_distribution='; '.join(f'{v:g}: {n}' for v,n in sorted(vals.items()))))
    counts=Counter((r['question_id'],r['label'] or '(unlabeled)') for r in rows)
    labels=[dict(question_id=q,label=l,count=n) for (q,l),n in counts.items()]
    semantic=[]
    if embeddings is not None:
        if len(embeddings)!=len(questions): raise ValueError('Embedding count mismatch')
        for j,q in enumerate(questions):
            sims=[(sum(a*b for a,b in zip(embeddings[j],v)),k) for k,v in enumerate(embeddings) if k!=j]
            for sim,k in sorted(sims,reverse=True)[:2]:
                semantic.append(dict(question_id=q,related_question_id=questions[k],cosine_similarity=sim,
                    related_question_text=text[k]))
    return dict(summary=dict(students=len(students),questions=len(questions),responses=len(rows),
        model_fitted=bool(mu),model='Regularized fractional-logit / binary Rasch when all scores are binary'),
        warnings=warnings,students=student_out,questions=question_out,labels=labels,
        responses=rows,semantic_groups=semantic)


In [0]:
DEMO_CSV = 'Student,Question,Question text,Rubric answer/title (verbatim),Result,Points earned,Points possible,Points lost,Score (%)\r\nDemo student 01,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 01,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 01,Q3,Calculate 2/3 + 1/4.,,correct,8.0,8,0.0,100.0\r\nDemo student 01,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 01,Q5,Find the area of a triangle with base 8 and height 5.,,correct,10.0,10,0.0,100.0\r\nDemo student 01,Q6,Find the area of a circle with radius 3.,,minor mistake,9.0,12,3.0,75.0\r\nDemo student 02,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 02,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 02,Q3,Calculate 2/3 + 1/4.,,correct,8.0,8,0.0,100.0\r\nDemo student 02,Q4,Subtract 3/8 from 5/6.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 02,Q5,Find the area of a triangle with base 8 and height 5.,,minor mistake,7.5,10,2.5,75.0\r\nDemo student 02,Q6,Find the area of a circle with radius 3.,,minor mistake,9.0,12,3.0,75.0\r\nDemo student 03,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 03,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 03,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 03,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 03,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 03,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 04,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 04,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 04,Q3,Calculate 2/3 + 1/4.,,correct,8.0,8,0.0,100.0\r\nDemo student 04,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 04,Q5,Find the area of a triangle with base 8 and height 5.,,correct,10.0,10,0.0,100.0\r\nDemo student 04,Q6,Find the area of a circle with radius 3.,,minor mistake,9.0,12,3.0,75.0\r\nDemo student 05,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 05,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 05,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 05,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 05,Q5,Find the area of a triangle with base 8 and height 5.,,small misstep on step (c),2.5,10,7.5,25.0\r\nDemo student 05,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 06,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 06,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 06,Q3,Calculate 2/3 + 1/4.,,correct,8.0,8,0.0,100.0\r\nDemo student 06,Q4,Subtract 3/8 from 5/6.,,correct,8.0,8,0.0,100.0\r\nDemo student 06,Q5,Find the area of a triangle with base 8 and height 5.,,correct,10.0,10,0.0,100.0\r\nDemo student 06,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 07,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 07,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 07,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 07,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 07,Q5,Find the area of a triangle with base 8 and height 5.,,minor mistake,7.5,10,2.5,75.0\r\nDemo student 07,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 08,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 08,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 08,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 08,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 08,Q5,Find the area of a triangle with base 8 and height 5.,,minor mistake,7.5,10,2.5,75.0\r\nDemo student 08,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 09,Q1,Solve 3x + 7 = 22.,,small misstep on step (c),1.0,4,3.0,25.0\r\nDemo student 09,Q2,Solve 5y - 4 = 16.,,no credit,0.0,6,6.0,0.0\r\nDemo student 09,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 09,Q4,Subtract 3/8 from 5/6.,,no credit,0.0,8,8.0,0.0\r\nDemo student 09,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 09,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 10,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 10,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 10,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 10,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 10,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 10,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 11,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 11,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 11,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 11,Q4,Subtract 3/8 from 5/6.,,correct,8.0,8,0.0,100.0\r\nDemo student 11,Q5,Find the area of a triangle with base 8 and height 5.,,minor mistake,7.5,10,2.5,75.0\r\nDemo student 11,Q6,Find the area of a circle with radius 3.,,correct,12.0,12,0.0,100.0\r\nDemo student 12,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 12,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 12,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 12,Q4,Subtract 3/8 from 5/6.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 12,Q5,Find the area of a triangle with base 8 and height 5.,,minor mistake,7.5,10,2.5,75.0\r\nDemo student 12,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 13,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 13,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 13,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 13,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 13,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 13,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 14,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 14,Q2,Solve 5y - 4 = 16.,,small misstep on step (c),1.5,6,4.5,25.0\r\nDemo student 14,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 14,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 14,Q5,Find the area of a triangle with base 8 and height 5.,,small misstep on step (c),2.5,10,7.5,25.0\r\nDemo student 14,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 15,Q1,Solve 3x + 7 = 22.,,small misstep on step (c),1.0,4,3.0,25.0\r\nDemo student 15,Q2,Solve 5y - 4 = 16.,,no credit,0.0,6,6.0,0.0\r\nDemo student 15,Q3,Calculate 2/3 + 1/4.,,no credit,0.0,8,8.0,0.0\r\nDemo student 15,Q4,Subtract 3/8 from 5/6.,,no credit,0.0,8,8.0,0.0\r\nDemo student 15,Q5,Find the area of a triangle with base 8 and height 5.,,small misstep on step (c),2.5,10,7.5,25.0\r\nDemo student 15,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 16,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 16,Q2,Solve 5y - 4 = 16.,,small misstep on step (c),1.5,6,4.5,25.0\r\nDemo student 16,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 16,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 16,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 16,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 17,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 17,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 17,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 17,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 17,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 17,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 18,Q1,Solve 3x + 7 = 22.,,small misstep on step (c),1.0,4,3.0,25.0\r\nDemo student 18,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 18,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 18,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 18,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 18,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 19,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 19,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 19,Q3,Calculate 2/3 + 1/4.,,correct,8.0,8,0.0,100.0\r\nDemo student 19,Q4,Subtract 3/8 from 5/6.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 19,Q5,Find the area of a triangle with base 8 and height 5.,,minor mistake,7.5,10,2.5,75.0\r\nDemo student 19,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 20,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 20,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 20,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 20,Q4,Subtract 3/8 from 5/6.,,no credit,0.0,8,8.0,0.0\r\nDemo student 20,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 20,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 21,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 21,Q2,Solve 5y - 4 = 16.,,small misstep on step (c),1.5,6,4.5,25.0\r\nDemo student 21,Q3,Calculate 2/3 + 1/4.,,no credit,0.0,8,8.0,0.0\r\nDemo student 21,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 21,Q5,Find the area of a triangle with base 8 and height 5.,,small misstep on step (c),2.5,10,7.5,25.0\r\nDemo student 21,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 22,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 22,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 22,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 22,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 22,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 22,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 23,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 23,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 23,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 23,Q4,Subtract 3/8 from 5/6.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 23,Q5,Find the area of a triangle with base 8 and height 5.,,correct,10.0,10,0.0,100.0\r\nDemo student 23,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 24,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 24,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 24,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 24,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 24,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 24,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 25,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 25,Q2,Solve 5y - 4 = 16.,,small misstep on step (c),1.5,6,4.5,25.0\r\nDemo student 25,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 25,Q4,Subtract 3/8 from 5/6.,,no credit,0.0,8,8.0,0.0\r\nDemo student 25,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 25,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 26,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 26,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 26,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 26,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 26,Q5,Find the area of a triangle with base 8 and height 5.,,minor mistake,7.5,10,2.5,75.0\r\nDemo student 26,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 27,Q1,Solve 3x + 7 = 22.,,small misstep on step (c),1.0,4,3.0,25.0\r\nDemo student 27,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 27,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 27,Q4,Subtract 3/8 from 5/6.,,no credit,0.0,8,8.0,0.0\r\nDemo student 27,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 27,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 28,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 28,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 28,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 28,Q4,Subtract 3/8 from 5/6.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 28,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 28,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 29,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 29,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 29,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 29,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 29,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 29,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 30,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 30,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 30,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 30,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 30,Q5,Find the area of a triangle with base 8 and height 5.,,small misstep on step (c),2.5,10,7.5,25.0\r\nDemo student 30,Q6,Find the area of a circle with radius 3.,,minor mistake,9.0,12,3.0,75.0\r\nDemo student 31,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 31,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 31,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 31,Q4,Subtract 3/8 from 5/6.,,correct,8.0,8,0.0,100.0\r\nDemo student 31,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 31,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 32,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 32,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 32,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 32,Q4,Subtract 3/8 from 5/6.,,correct,8.0,8,0.0,100.0\r\nDemo student 32,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 32,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\nDemo student 33,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 33,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 33,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 33,Q4,Subtract 3/8 from 5/6.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 33,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 33,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 34,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 34,Q2,Solve 5y - 4 = 16.,,2 minor mistakes or a major mistake,3.0,6,3.0,50.0\r\nDemo student 34,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 34,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 34,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 34,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 35,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 35,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 35,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 35,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 35,Q5,Find the area of a triangle with base 8 and height 5.,,small misstep on step (c),2.5,10,7.5,25.0\r\nDemo student 35,Q6,Find the area of a circle with radius 3.,,minor mistake,9.0,12,3.0,75.0\r\nDemo student 36,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 36,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 36,Q3,Calculate 2/3 + 1/4.,,correct,8.0,8,0.0,100.0\r\nDemo student 36,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 36,Q5,Find the area of a triangle with base 8 and height 5.,,no credit,0.0,10,10.0,0.0\r\nDemo student 36,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 37,Q1,Solve 3x + 7 = 22.,,minor mistake,3.0,4,1.0,75.0\r\nDemo student 37,Q2,Solve 5y - 4 = 16.,,small misstep on step (c),1.5,6,4.5,25.0\r\nDemo student 37,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 37,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 37,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 37,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 38,Q1,Solve 3x + 7 = 22.,,small misstep on step (c),1.0,4,3.0,25.0\r\nDemo student 38,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 38,Q3,Calculate 2/3 + 1/4.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 38,Q4,Subtract 3/8 from 5/6.,,minor mistake,6.0,8,2.0,75.0\r\nDemo student 38,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 38,Q6,Find the area of a circle with radius 3.,,small misstep on step (c),3.0,12,9.0,25.0\r\nDemo student 39,Q1,Solve 3x + 7 = 22.,,2 minor mistakes or a major mistake,2.0,4,2.0,50.0\r\nDemo student 39,Q2,Solve 5y - 4 = 16.,,minor mistake,4.5,6,1.5,75.0\r\nDemo student 39,Q3,Calculate 2/3 + 1/4.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 39,Q4,Subtract 3/8 from 5/6.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 39,Q5,Find the area of a triangle with base 8 and height 5.,,2 minor mistakes or a major mistake,5.0,10,5.0,50.0\r\nDemo student 39,Q6,Find the area of a circle with radius 3.,,no credit,0.0,12,12.0,0.0\r\nDemo student 40,Q1,Solve 3x + 7 = 22.,,correct,4.0,4,0.0,100.0\r\nDemo student 40,Q2,Solve 5y - 4 = 16.,,correct,6.0,6,0.0,100.0\r\nDemo student 40,Q3,Calculate 2/3 + 1/4.,,small misstep on step (c),2.0,8,6.0,25.0\r\nDemo student 40,Q4,Subtract 3/8 from 5/6.,,2 minor mistakes or a major mistake,4.0,8,4.0,50.0\r\nDemo student 40,Q5,Find the area of a triangle with base 8 and height 5.,,small misstep on step (c),2.5,10,7.5,25.0\r\nDemo student 40,Q6,Find the area of a circle with radius 3.,,2 minor mistakes or a major mistake,6.0,12,6.0,50.0\r\n'
rows=parse_csv(Path(CSV_PATH).read_text(encoding='utf-8-sig') if CSV_PATH else DEMO_CSV)
question_ids=list(dict.fromkeys(r['question_id'] for r in rows))
question_texts=[next(r['question_text'] for r in rows if r['question_id']==q) for q in question_ids]
print(f'Loaded {len(rows)} responses. Synthetic data: {not bool(CSV_PATH)}')


Loaded 240 responses. Synthetic data: True


In [0]:
question_embeddings=None
if EMBED_QUESTIONS:
    from databricks.sdk import WorkspaceClient
    client=WorkspaceClient()
    question_embeddings=[]
    for text in question_texts:
        response=client.serving_endpoints.query(name=EMBEDDING_ENDPOINT,input=text)
        v=response.data[0].embedding
        norm=math.sqrt(sum(a*a for a in v))
        if not norm or not all(math.isfinite(a) for a in v): raise ValueError('Invalid embedding')
        question_embeddings.append([a/norm for a in v])
    print('Embedded',len(question_embeddings),'unique questions')
else:
    print('Embeddings disabled; score and label summaries still run.')


Embedded 6 unique questions


In [0]:
result=analyze(rows,fit_rasch,question_embeddings)
result['summary']['synthetic']=not bool(CSV_PATH)
result['summary']['embedding_endpoint']=EMBEDDING_ENDPOINT if EMBED_QUESTIONS else None
student_statistics=result['students']
question_statistics=result['questions']
label_statistics=result['labels']
response_details=result['responses']
print(json.dumps(result['summary'],indent=2))
for warning in result['warnings']: print('NOTE:',warning)
if 'displayHTML' in globals():
    import html
    output='<h2>Assessment analysis</h2>'
    for key in ['students','questions','labels','responses','semantic_groups']:
        records=result[key]
        if not records: continue
        output+='<h3>'+key+'</h3><div style="max-height:350px;overflow:auto"><table border="1"><tr>'
        cols=list(records[0]);output+=''.join('<th>'+html.escape(c)+'</th>' for c in cols)+'</tr>'
        for r in records[:200]:
            output+='<tr>'+''.join('<td>'+html.escape(str(r.get(c,'')))+'</td>' for c in cols)+'</tr>'
        output+='</table></div>'
    displayHTML(output)


{
  "students": 40,
  "questions": 6,
  "responses": 240,
  "model_fitted": true,
  "model": "Regularized fractional-logit / binary Rasch when all scores are binary",
  "synthetic": true,
  "embedding_endpoint": "databricks-gte-large-en"
}
NOTE: Fractional-logit mean model with Gaussian regularization; not a partial-credit IRT category likelihood.
NOTE: Expected score is model-based, not a corrected grade. No uncertainty intervals.
NOTE: Expected totals cover only the modeled question subset; missing responses are omitted from fitting.
NOTE: Rubric labels are preserved literally; no automatic misconception or severity inference.
NOTE: Question embeddings drive related-question summaries, not assumed difficulty or proficiency.


student,observed_points,graded_questions,proficiency,estimated_true_score,modeled_max_points,modeled_questions
Demo student 01,40.0,6,0.7980907768301477,31.949310922399892,48.0,6
Demo student 02,36.5,6,0.573331087009804,29.619093392068738,48.0,6
Demo student 03,23.5,6,0.038165801834336925,23.728041434268828,48.0,6
Demo student 04,38.0,6,0.6846353559945427,30.789015494511546,48.0,6
Demo student 05,21.5,6,-0.17122212998586817,21.39940153801266,48.0,6
Demo student 06,42.0,6,1.0328673580171388,34.227127552690334,48.0,6
Demo student 07,27.0,6,0.1433190825050605,24.902971316714257,48.0,6
Demo student 08,22.5,6,0.038165801834336925,23.728041434268828,48.0,6
Demo student 09,3.0,6,-1.1521121063136497,11.669926212798064,48.0,6
Demo student 10,14.0,6,-0.48695810035448595,17.99048520306623,48.0,6


In [0]:
# For Jobs API use return_json=true. Interactive notebook defaults to no exit.
payload=json.dumps(result,ensure_ascii=False,allow_nan=False)
if RETURN_JSON:
    if len(payload.encode('utf-8'))>4_500_000:
        raise ValueError('Result too large for Jobs output. Persist JSON/tables in a Volume instead.')
    dbutils.notebook.exit(payload)
else:
    print('Result is available as result; set return_json=true for API output.')


Result is available as result; set return_json=true for API output.
